<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/MINI_CHATGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf requests==2.32.4


In [ ]:
import os
from google.colab import files

# Document Loaders
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)
# Text Splitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

# OpenAI Models & Embeddings
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)
# Vector Store
from langchain_community.vectorstores import FAISS

# RAG Utilities
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("MINI_CHAT_KEY")


In [ ]:
print("Upload PDF / TXT / MD files")
uploaded = files.upload()

In [ ]:
documents = []

for file in uploaded.keys():

    if file.endswith(".pdf"):
        loader = PyPDFLoader(file)
        documents.extend(loader.load())

    elif file.endswith(".txt") or file.endswith(".md"):
        loader = TextLoader(file)
        documents.extend(loader.load())

print("Documents Loaded:", len(documents))

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
# =============================
# STEP 6 — Create Vector DB (SAFE)
# =============================

# ✅ Validate chunks
if not chunks:
    raise ValueError("❌ No chunks created. Uploaded file may be empty or unreadable.")

# ✅ Remove empty text chunks (CRITICAL)
chunks = [c for c in chunks if c.page_content and c.page_content.strip()]

if not chunks:
    raise ValueError("❌ All chunks are empty after cleaning. Cannot build Vector DB.")

print("Valid chunks for embeddings:", len(chunks))

# ✅ Create FAISS vector DB
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ Vector DB Ready")



In [ ]:

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.
If unsure, say you don't know.

Context:
{context}

Question:
{input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

qa_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=document_chain
)

print("\n✅ Knowledge Assistant Ready!")
print("Type 'exit' to stop.\n")


In [ ]:
print("\n✅ Knowledge Assistant Ready!")
print("Type 'exit' to stop.\n")

while True:
    query = input("Ask Question: ")

    if query.lower() == "exit":
        print("Session Ended")
        break

    response = qa_chain.invoke({"input": query})

    print("\nAnswer:\n", response["answer"])